## Clasificador de movimientos con MLP ##

### Imports ###

In [26]:
import os
import numpy as np
import pandas as pd
from scipy.signal import butter,sosfilt,sosfilt_zi
from sklearn.model_selection import train_test_split
from collections import Counter
from scipy.stats import skew,kurtosis
from scipy.fft import rfft,rfftfreq

DATA_DIR = "data"
CLASSES = ["quieto","brazo","salto","sentadilla"]

FS = 50
WINDOW_SIZE = 70
CUTOFF = 7.5
ORDER = 2

SIGNALS = ["ax","ay","az","gx","gy","gz","a_mag","g_mag"]

### Carga, etiquetado y split de los datos ###

In [3]:
records=[]

for label in CLASSES:
    folder=os.path.join(DATA_DIR,label)
    for file in sorted(os.listdir(folder)):
        if file.endswith(".csv"):
            records.append({"path":os.path.join(folder,file),"label":label})
                    
print(f"Grabaciones: {len(records)}")

Grabaciones: 120


In [5]:
train_records, test_records = train_test_split(records, test_size=0.2, random_state=42, stratify=[r["label"] for r in records])

print("Train:",len(train_records))
print("Train:",Counter(r["label"] for r in train_records))
print("Test:",len(test_records))
print("Test:",Counter(r["label"] for r in test_records))

Train: 96
Train: Counter({'salto': 24, 'brazo': 24, 'sentadilla': 24, 'quieto': 24})
Test: 24
Test: Counter({'salto': 6, 'sentadilla': 6, 'quieto': 6, 'brazo': 6})


In [10]:
def add_magnitudes(df):
    df["a_mag"] = np.sqrt(df["ax"]**2+df["ay"]**2+df["az"]**2)
    df["g_mag"] = np.sqrt(df["gx"]**2+df["gy"]**2+df["gz"]**2)
    return df

def load_record(record):
    df=pd.read_csv(record["path"])
    df=add_magnitudes(df)
    return df

train_data = [{"df":load_record(r),"label":r["label"],"path":r["path"]} for r in train_records]
test_data = [{"df":load_record(r),"label":r["label"],"path":r["path"]} for r in test_records]

In [13]:
sos = butter(ORDER, CUTOFF, btype="lowpass", fs=FS, output="sos")

def filter_recording(df,sos):
    df=df.copy()

    for signal in SIGNALS:
        x=df[signal].to_numpy()
        zi=sosfilt_zi(sos)*x[0]
        df[signal],_=sosfilt(sos,x,zi=zi)

    return df

for record in train_data:
    record["df"]=filter_recording(record["df"],sos)

for record in test_data:
    record["df"]=filter_recording(record["df"],sos)

In [ ]:
def create_windows(data):
    X=[]
    y=[]

    for record in data:
        x=record["df"][SIGNALS].to_numpy()
        n=len(x)//WINDOW_SIZE

        for i in range(n):
            window=x[i*WINDOW_SIZE:(i+1)*WINDOW_SIZE]
            X.append(window)
            y.append(record["label"])

    return np.array(X),np.array(y)

X_train, y_train = create_windows(train_data)
X_test, y_test = create_windows(test_data)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (192, 70, 8)
y_train: (192,)
X_test: (48, 70, 8)
y_test: (48,)


In [27]:
def extract_features(X, fs=FS):
    features=[]

    for window in X:
        row=[]

        for j in range(window.shape[1]):
            x=window[:,j]
            x_centered=x-x.mean()

            spectrum=np.abs(rfft(x_centered))
            freqs=rfftfreq(len(x),1/fs)

            if len(spectrum)>1:
                peak_freq=freqs[1+np.argmax(spectrum[1:])]
            else:
                peak_freq=0

            row.extend([
                np.mean(x),
                np.std(x),
                np.min(x),
                np.max(x),
                np.ptp(x),
                np.median(x),
                np.percentile(x,75)-np.percentile(x,25),
                np.median(np.abs(x-np.median(x))),
                np.sqrt(np.mean(x**2)),
                np.mean(np.abs(x)),
                skew(x),
                kurtosis(x),
                np.mean(x_centered[1:]*x_centered[:-1]<0),
                peak_freq
            ])

        features.append(row)

    return np.array(features)

X_train_feat = extract_features(X_train)
X_test_feat = extract_features(X_test)

print(X_train_feat.shape)
print(X_test_feat.shape)

(192, 112)
(48, 112)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train=scaler.fit_transform(X_train_feat)
X_test=scaler.transform(X_test_feat)
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier

model=MLPClassifier(
    hidden_layer_sizes=(100,50),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42
)

model.fit(X_train,y_train)

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy=accuracy_score(y_test,y_pred)

print(f"Accuracy: {accuracy:.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm=confusion_matrix(
    y_test,
    y_pred,
    labels=CLASSES
)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.xticks(range(len(CLASSES)),CLASSES,rotation=45)
plt.yticks(range(len(CLASSES)),CLASSES)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de confusión")

for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j,i,cm[i,j],ha="center",va="center")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    target_names=CLASSES
))